## Encontro 3 - Features temporais


## Importar e organizar os dados


In [3]:
import pandas as pd

df = pd.read_csv("../dados/mecaniqa_dataset.csv")

df.columns = df.columns.str.strip()
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index().asfreq("D")

colunas = ["Trocas_Oleo", "Manutencao_Motor"]
df = df[colunas]

df.head()

,Trocas_Oleo,Manutencao_Motor
Data,,
2024-01-01,11.0,2.0
2024-01-02,9.0,4.0
2024-01-03,12.0,1.0
2024-01-04,15.0,8.0
2024-01-05,25.0,10.0


## Tratar os dados


In [4]:
df["Trocas_Oleo"] = df["Trocas_Oleo"].clip(upper=55)
df["Manutencao_Motor"] = df["Manutencao_Motor"].clip(lower=0, upper=26)

observado = df.copy()

# Preenche o histórico usando somente valores anteriores.
df = df.ffill()

df.isna().sum()

Trocas_Oleo         0
Manutencao_Motor    0
dtype: int64

## Criar os lags e a média móvel

In [5]:
df_features = pd.DataFrame(index=df.index)

for coluna in colunas:
    df_features[f"{coluna}_lag_1"] = df[coluna].shift(1)
    df_features[f"{coluna}_lag_7"] = df[coluna].shift(7)
    df_features[f"{coluna}_lag_30"] = df[coluna].shift(30)

    df_features[f"{coluna}_media_7"] = (
        df[coluna].shift(1).rolling(7).mean()
    )

## Remover os vazios e conferir as primeiras 15 linhas

In [6]:
df_features = df_features.dropna()

print("Quantidade de linhas:", len(df_features))
print("Valores ausentes:", df_features.isna().sum().sum())

df_features.head(15)

Quantidade de linhas: 701
Valores ausentes: 0


,Trocas_Oleo_lag_1,Trocas_Oleo_lag_7,Trocas_Oleo_lag_30,Trocas_Oleo_media_7,Manutencao_Motor_lag_1,Manutencao_Motor_lag_7,Manutencao_Motor_lag_30,Manutencao_Motor_media_7
Data,,,,,,,,
2024-01-31,14.0,10.0,11.0,19.571429,4.0,2.0,2.0,7.714286
2024-02-01,14.0,13.0,9.0,20.142857,5.0,10.0,4.0,8.142857
2024-02-02,21.0,30.0,12.0,21.285714,13.0,12.0,1.0,8.571429
2024-02-03,31.0,26.0,15.0,21.428571,12.0,10.0,8.0,8.571429
2024-02-04,28.0,31.0,25.0,21.714286,13.0,10.0,10.0,9.000000
2024-02-05,34.0,13.0,25.0,22.142857,12.0,6.0,10.0,9.285714
2024-02-06,13.0,14.0,31.0,22.142857,5.0,4.0,14.0,9.142857
2024-02-07,17.0,14.0,13.0,22.571429,10.0,5.0,8.0,10.000000
2024-02-08,11.0,21.0,10.0,22.142857,3.0,13.0,6.0,9.714286


## Criar e salvar as bases com os alvos

In [8]:
base_oleo = df_features.join(
    observado["Trocas_Oleo"].rename("Alvo")
).dropna()

base_motor = df_features.join(
    observado["Manutencao_Motor"].rename("Alvo")
).dropna()

base_oleo.to_csv("base_oat2_oleo.csv", index_label="Data")
base_motor.to_csv("base_oat2_motor.csv", index_label="Data")

print("Base de óleo:", base_oleo.shape)
print("Base de motor:", base_motor.shape)

Base de óleo: (695, 9)
Base de motor: (698, 9)
